In [1]:
from phonemize import phonemize
from transformers import AutoTokenizer
from dataloader import FilePathDataset, build_dataloader, Collater
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("GoToCompany/Llama-Sahabat-AI-v2-70B-IT")

# Sample texts
texts = [
    "pemograman adalah seni dalam menulis, seni dalam menyusun logika, dan seni dalam menciptakan solusi.",
    "Suatu hari, Bujang Enok membunuh seekor ular berbisa yang telah meresahkan banyak orang.",
    "machine learning sangat menarik untuk dipelajari.",
    "indonesia adalah negara yang indah sekali.",
    "teknologi artificial intelligence berkembang pesat di era modern ini."
]

# Process texts
processed_data = []
for i, text in enumerate(texts):
    print(f"Processing text {i+1}/{len(texts)}: {text[:50]}...")
    result = phonemize(text, tokenizer)
    processed_data.append({
        'input_ids': result['input_ids'],
        'phonemes': result['phonemes']
    })
    
# Create FilePathDataset
dataloader_dataset = FilePathDataset(
    dataset=processed_data,
    tokenizer_name="GoToCompany/Llama-Sahabat-AI-v2-70B-IT",
    token_separator="|",
    token_mask="M",
    max_mel_length=512,
    word_mask_prob=0.3,
    phoneme_mask_prob=0.5,
    replace_prob=0.3
)

# Test access
print(f"Dataset size: {len(dataloader_dataset)}")

# Access first item
item = dataloader_dataset[1]
phonemes_tensor, words_tensor, labels_tensor, masked_index = item

print(f"First item:")
print(f"  Phonemes shape: {phonemes_tensor.shape}")
print(f"  Words shape: {words_tensor.shape}")
print(f"  Labels shape: {labels_tensor.shape}")
print(f"  Masked indices: {masked_index}")

/workspace/PL-BERT-v2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing text 1/5: pemograman adalah seni dalam menulis, seni dalam m...
Tokenized 14 words → 14 encoded
   1. pemograman → ['p', 'em', 'ogram', 'an']
   2. adalah → ['ad', 'alah']
   3. seni → ['sen', 'i']
   4. dalam → ['d', 'alam']
   5. menulis → ['men', 'ulis']
   6. seni → ['sen', 'i']
   7. dalam → ['d', 'alam']
   8. menyusun → ['men', 'y', 'us', 'un']
   9. logika → ['log', 'ika']
  10. dan → ['dan']
Processing text 2/5: Suatu hari, Bujang Enok membunuh seekor ular berbi...
Tokenized 13 words → 13 encoded
   1. suatu → ['su', 'atu']
   2. hari → ['hari']
   3. bujang → ['b', 'uj', 'ang']
   4. enok → ['en', 'ok']
   5. membunuh → ['m', 'emb', 'un', 'uh']
   6. seekor → ['seek', 'or']
   7. ular → ['ular']
   8. berbisa → ['ber', 'b', 'isa']
   9. yang → ['yang']
  10. telah → ['tel', 'ah']
Processing text 3/5: machine learning sangat menarik untuk dipelajari....
Tokenized 6 words → 6 encoded
   1. machine → ['machine']
   2. learning → ['learning']
   3. sangat → ['s', 'ang'

In [2]:
collater = Collater(tokenizer=tokenizer, debug=True)

# Get a few items manually
batch_items = []
for i in range(3):  # Ambil 3 item pertama
    item = dataloader_dataset[i]
    batch_items.append(item)
    print(f"\nItem {i}: shapes = {[x.shape if hasattr(x, 'shape') else len(x) for x in item]}")

# Manual collate
print("\n--- MANUAL COLLATING ---")
collated = collater(batch_items)
words, labels, phonemes, input_lengths, masked_indices = collated

print(f"\n📦 COLLATED BATCH RESULTS:")
print(f"  Words shape:    {words.shape}")
print(f"  Labels shape:   {labels.shape}")
print(f"  Phonemes shape: {phonemes.shape}")
print(f"  Input lengths:  {input_lengths}")
print(f"  Masked indices: {[len(m) for m in masked_indices]}")


=== [IDX 0] INPUT DATA ===
Phonemes (word-level): ['pəmoɡrˈaman', 'adˈalah', 'sˈɛni', 'dˈalam', 'mənˈulis', 'sˈɛni', 'dˈalam', 'məɲˈusun', 'loɡˈika', 'dˈan', 'sˈɛni', 'dˈalam', 'məntʃiptˈakan', 'solˈusi']
Input IDs (per-word BPE): [[79, 336, 13255, 276], [329, 30493], [12021, 72], [67, 17243], [5794, 65130], [12021, 72], [67, 17243], [5794, 88, 355, 359], [848, 11755], [36255], [12021, 72], [67, 17243], [76, 967, 11696, 19818], [39298, 53913]]

--- Word 1 ---
  phoneme_word : pəmoɡrˈaman
  bpe_ids      : [79, 336, 13255, 276]
  subword_tokens : ['p', 'em', 'ogram', 'an']
  -> phoneme kept as is

--- Word 2 ---
  phoneme_word : adˈalah
  bpe_ids      : [329, 30493]
  subword_tokens : ['ad', 'alah']
  -> phoneme masked: MMMMMMM

--- Word 3 ---
  phoneme_word : sˈɛni
  bpe_ids      : [12021, 72]
  subword_tokens : ['sen', 'i']
  -> phoneme kept as is

--- Word 4 ---
  phoneme_word : dˈalam
  bpe_ids      : [67, 17243]
  subword_tokens : ['d', 'alam']
  -> phoneme kept as is

--- Word 5 -

In [5]:
dataloader = build_dataloader(
    df=processed_data,
    validation=False,
    batch_size=2,
    num_workers=0,  # Set to 0 untuk debugging
    device='cpu',
    collate_config={
        'debug': True,  # Enable debug di collater
        'return_wave': False
    },
    dataset_config={
        "tokenizer_name": "GoToCompany/Llama-Sahabat-AI-v2-70B-IT",
        "token_separator": "|",
        "token_mask": "M",
        "max_mel_length": 512,
        "word_mask_prob": 0.3,
        "phoneme_mask_prob": 0.5,
        "replace_prob": 0.3
    }
)

print(f"\n🚀 DataLoader created with:")
print(f"  Dataset size: {len(dataloader.dataset)}")
print(f"  Batch size: {dataloader.batch_size}")
print(f"  Num batches: {len(dataloader)}")

# =============================================================================
# 3. ITERATE THROUGH BATCHES
# =============================================================================
print("\n" + "="*60)
print("3. ITERATING THROUGH BATCHES")
print("="*60)

# ...existing code...

# ...existing code...

for batch_idx, (words, labels, phonemes, input_lengths, masked_indices) in enumerate(dataloader):
    print(f"\n🔄 BATCH {batch_idx + 1}:")
    print(f"  Words shape:      {words.shape}")
    print(f"  Labels shape:     {labels.shape}")
    print(f"  Phonemes shape:   {phonemes.shape}")
    print(f"  Input lengths:    {input_lengths}")
    print(f"  Masked indices:   {[len(m) for m in masked_indices]}")
    
    # Decode beberapa contoh untuk verifikasi
    print(f"\n  📝 Sample decoded words (first sequence):")
    first_words = words[0]
    
    # ✅ Handle pad token properly
    if hasattr(tokenizer, 'pad_token_id') and tokenizer.pad_token_id is not None:
        non_padded = first_words[first_words != tokenizer.pad_token_id]
    else:
        # Fallback: use sequence length or just take first N tokens
        actual_length = input_lengths[0] if len(input_lengths) > 0 else len(first_words)
        non_padded = first_words[:actual_length]
    
    # Convert to list and decode
    try:
        decoded = tokenizer.decode(non_padded.tolist(), skip_special_tokens=True)
        print(f"     '{decoded[:100]}...'")
    except Exception as e:
        print(f"     [Decode error: {e}]")
        print(f"     Raw token IDs: {non_padded.tolist()[:10]}...")
    
    print(f"\n  📱 Phonemes (first 20 chars): {phonemes[0][:20].tolist()}")
    print(f"  🏷️  Labels (first 20 chars):   {labels[0][:20].tolist()}")
    
    if batch_idx >= 1:  # Hanya 2 batch pertama
        break

# ...existing code...

# ...existing code...


177
[INFO] Tokenizer loaded: GoToCompany/Llama-Sahabat-AI-v2-70B-IT
[INFO] Vocab size: 128256
[INFO] EOS token ID used as word separator: 128009

🚀 DataLoader created with:
  Dataset size: 5
  Batch size: 2
  Num batches: 2

3. ITERATING THROUGH BATCHES

=== [IDX 0] INPUT DATA ===
Phonemes (word-level): ['pəmoɡrˈaman', 'adˈalah', 'sˈɛni', 'dˈalam', 'mənˈulis', 'sˈɛni', 'dˈalam', 'məɲˈusun', 'loɡˈika', 'dˈan', 'sˈɛni', 'dˈalam', 'məntʃiptˈakan', 'solˈusi']
Input IDs (per-word BPE): [[79, 336, 13255, 276], [329, 30493], [12021, 72], [67, 17243], [5794, 65130], [12021, 72], [67, 17243], [5794, 88, 355, 359], [848, 11755], [36255], [12021, 72], [67, 17243], [76, 967, 11696, 19818], [39298, 53913]]

--- Word 1 ---
  phoneme_word : pəmoɡrˈaman
  bpe_ids      : [79, 336, 13255, 276]
  subword_tokens : ['p', 'em', 'ogram', 'an']
  -> phoneme kept as is

--- Word 2 ---
  phoneme_word : adˈalah
  bpe_ids      : [329, 30493]
  subword_tokens : ['ad', 'alah']
  -> phoneme kept as is

--- Word 3 --

In [6]:

# =============================================================================
# 4. VALIDATION DATALOADER
# =============================================================================
print("\n" + "="*60)
print("4. TESTING VALIDATION DATALOADER")
print("="*60)

val_dataloader = build_dataloader(
    df=processed_data,
    validation=True,  # No shuffling, no drop_last
    batch_size=3,
    num_workers=0,
    device='cpu',
    collate_config={'debug': False},
    dataset_config={
        "tokenizer_name": "GoToCompany/Llama-Sahabat-AI-v2-70B-IT",
        "max_mel_length": 256,  # Shorter for validation
        "word_mask_prob": 0.0,  # No masking for validation
        "phoneme_mask_prob": 0.0,
        "replace_prob": 0.0
    }
)

print(f"📊 Validation DataLoader:")
print(f"  Shuffle: {val_dataloader.sampler is None}")
print(f"  Drop last: {val_dataloader.drop_last}")
print(f"  Batch size: {val_dataloader.batch_size}")

# Test first validation batch
val_batch = next(iter(val_dataloader))
words_val, labels_val, phonemes_val, lengths_val, masked_val = val_batch

print(f"\n✅ Validation batch:")
print(f"  Words shape: {words_val.shape}")
print(f"  No masking: {all(len(m) == 0 for m in masked_val)}")



4. TESTING VALIDATION DATALOADER
177
[INFO] Tokenizer loaded: GoToCompany/Llama-Sahabat-AI-v2-70B-IT
[INFO] Vocab size: 128256
[INFO] EOS token ID used as word separator: 128009
📊 Validation DataLoader:
  Shuffle: False
  Drop last: False
  Batch size: 3

=== [IDX 0] INPUT DATA ===
Phonemes (word-level): ['pəmoɡrˈaman', 'adˈalah', 'sˈɛni', 'dˈalam', 'mənˈulis', 'sˈɛni', 'dˈalam', 'məɲˈusun', 'loɡˈika', 'dˈan', 'sˈɛni', 'dˈalam', 'məntʃiptˈakan', 'solˈusi']
Input IDs (per-word BPE): [[79, 336, 13255, 276], [329, 30493], [12021, 72], [67, 17243], [5794, 65130], [12021, 72], [67, 17243], [5794, 88, 355, 359], [848, 11755], [36255], [12021, 72], [67, 17243], [76, 967, 11696, 19818], [39298, 53913]]

--- Word 1 ---
  phoneme_word : pəmoɡrˈaman
  bpe_ids      : [79, 336, 13255, 276]
  subword_tokens : ['p', 'em', 'ogram', 'an']
  -> phoneme kept as is

--- Word 2 ---
  phoneme_word : adˈalah
  bpe_ids      : [329, 30493]
  subword_tokens : ['ad', 'alah']
  -> phoneme kept as is

--- Word 3 

In [7]:

# =============================================================================
# 5. GPU TESTING (IF AVAILABLE)
# =============================================================================
if torch.cuda.is_available():
    print("\n" + "="*60)
    print("5. TESTING GPU DATALOADER")
    print("="*60)
    
    gpu_dataloader = build_dataloader(
        df=processed_data[:3],  # Smaller dataset for GPU test
        validation=False,
        batch_size=2,
        num_workers=0,
        device='cuda',
        collate_config={'debug': False},
        dataset_config={
            "tokenizer_name": "GoToCompany/Llama-Sahabat-AI-v2-70B-IT",
            "max_mel_length": 256,
        }
    )
    
    gpu_batch = next(iter(gpu_dataloader))
    print(f"🔥 GPU batch loaded successfully!")
    print(f"  Device check: {gpu_batch[0].device}")
else:
    print("\n❌ GPU not available, skipping GPU test")

print("\n🎉 ALL TESTS COMPLETED!")


5. TESTING GPU DATALOADER
177
[INFO] Tokenizer loaded: GoToCompany/Llama-Sahabat-AI-v2-70B-IT
[INFO] Vocab size: 128256
[INFO] EOS token ID used as word separator: 128009

=== [IDX 1] INPUT DATA ===
Phonemes (word-level): ['suˈatu', 'hˈari', 'bˈudʒaŋ', 'ˈɛnok', 'məmbˈunuh', 'səˈɛkɔr', 'ˈuːlɚ', 'bərbˈisa', 'jˈaŋ', 'təlˈah', 'mərəsˈahkan', 'bˈaɲak', 'ˈɔraŋ']
Input IDs (per-word BPE): [[28149, 36409], [77007], [65, 9832, 526], [268, 564], [76, 9034, 359, 12825], [26797, 269], [1299], [655, 65, 10994], [41345], [23774, 1494], [1195, 288, 75386], [65, 51403], [85268]]

--- Word 1 ---
  phoneme_word : suˈatu
  bpe_ids      : [28149, 36409]
  subword_tokens : ['su', 'atu']
  -> phoneme kept as is

--- Word 2 ---
  phoneme_word : hˈari
  bpe_ids      : [77007]
  subword_tokens : ['hari']
  -> phoneme kept as is

--- Word 3 ---
  phoneme_word : bˈudʒaŋ
  bpe_ids      : [65, 9832, 526]
  subword_tokens : ['b', 'uj', 'ang']
  -> phoneme kept as is

--- Word 4 ---
  phoneme_word : ˈɛnok
  bpe_ids